# Neuropixel open field: load and inspect

**Experiment.** A mouse with a chronic Neuropixels probe explores a glass open-field arena. Social stimuli (male, female, fake mouse) sit outside the arena, so the mouse "interacts" with them through the glass. One overhead IR camera records the arena.

**Files in `preliminary data /Neuropixel_open_field`**

| file | what it is |
|---|---|
| `spike_times.npy` | sample index of every spike (int64, 30 kHz) |
| `spike_clusters.npy` | cluster id of every spike (int32) |
| `cluster_info.tsv` | one row per cluster: Kilosort labels, Bombcell quality metrics, waveform shape, channel/depth, and the manual curation label `rf_label` |
| `cluster_rf.tsv` | `cluster_id -> rf_label` only (identical to the column in `cluster_info.tsv`) |
| `video_compressed.mkv` | overhead camera, 2148x2064 px, ~60 fps |

**Known caveats found while inspecting** (details in the last section):
* Ephys is 95.1 min, video is 117.0 min. Both start together (the camera kept running ~22 min after the ephys stopped), but the video's constant 59.637 fps timeline runs 0.6% fast relative to the ephys clock, so an event at video minute 90 is at ephys minute 90 plus 33 s. Section 6 measures this and fits a linear correction; `npx_io.video_to_ephys_time` applies it.
* No pose tracking yet (DLC/SLEAP), so no position, speed, or stimulus-approach events.
* Stimulus locations around the arena and the probe's target region are not recorded in the folder.

Helper functions live in `npx_io.py` next to this notebook.

In [ ]:
%matplotlib inline
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd()))
import npx_io as io

DATA_DIR = Path('../../preliminary data /Neuropixel_open_field').resolve()
CACHE = Path('cache'); CACHE.mkdir(exist_ok=True)
FS = io.FS
sns.set_context('notebook'); sns.set_style('ticks')
LABEL_COLORS = {'good': '#2a9d8f', 'mua': '#e9c46a', 'nonso': '#8e7dbe', 'noise': '#bbbbbb'}
assert DATA_DIR.exists(), DATA_DIR
sorted(p.name for p in DATA_DIR.iterdir())

## 1. Cluster metadata

`cluster_info.tsv` carries three independent quality opinions:
* `KSLabel`: Kilosort's own good/mua call.
* `bc_unitType` / `bombcell`: Bombcell's automatic classification (GOOD / MUA / NOISE / NON-SOMA).
* `rf_label`: the manual curation label (good / mua / nonso / noise). This is the one we use to select units.

In [ ]:
ci = io.load_cluster_info(DATA_DIR)
print(f'{len(ci)} clusters, {ci.n_spikes.sum():,} spikes, '
      f'{ci.useTheseTimesStop.max()/60:.1f} min recording')
display(ci['label'].value_counts().rename('n clusters').to_frame().T)
display(pd.crosstab(ci['rf_label'], ci['bc_unitType'], margins=True))
display(pd.crosstab(ci['rf_label'], ci['KSLabel'], margins=True))

In [ ]:
# The columns we will lean on most
cols = ['label', 'KSLabel', 'bc_unitType', 'ch', 'sh', 'depth_um', 'fr_hz', 'n_spikes',
        'amp', 'rawAmplitude', 'signalToNoiseRatio', 'fractionRPVs_estimatedTauR',
        'presenceRatio', 'percentageSpikesMissing_gaussian', 'maxDriftEstimate',
        'duration_ms', 'nPeaks', 'nTroughs']
ci[cols].head(10)

### Where are the units on the probe?
Depth is in µm from the probe tip (Kilosort convention) and covers 720 to 3990 µm, so roughly 3.3 mm of the shank has units. `sh` is 0 or 1. Confirm with the recording notes which probe type this is and what brain region the span covers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), gridspec_kw=dict(width_ratios=[1.2, 1, 1]))
ax = axes[0]
for lab in io.LABEL_ORDER:
    d = ci[ci.label == lab]
    ax.scatter(d.fr_hz, d.depth_um, s=12, alpha=.7, c=LABEL_COLORS[lab], label=f'{lab} (n={len(d)})')
ax.set_xscale('log'); ax.set_xlabel('mean firing rate (Hz)'); ax.set_ylabel('depth from tip (µm)')
ax.legend(frameon=False, fontsize=9); ax.set_title('units along the probe')

ax = axes[1]
bins = np.arange(600, 4100, 100)
for lab in ['good', 'mua']:
    ax.hist(ci.depth_um[ci.label == lab], bins=bins, orientation='horizontal', alpha=.7,
            color=LABEL_COLORS[lab], label=lab)
ax.set_xlabel('n units / 100 µm'); ax.set_ylabel('depth (µm)'); ax.legend(frameon=False); ax.set_title('unit density')

ax = axes[2]
ax.scatter(ci.ch, ci.depth_um, s=8, c=ci.sh, cmap='coolwarm')
ax.set_xlabel('peak channel'); ax.set_ylabel('depth (µm)'); ax.set_title('channel vs depth (color = sh)')
sns.despine(); plt.tight_layout()

### Quality metrics
A quick look at how the manual labels relate to the Bombcell metrics. Useful for deciding whether to relax or tighten the unit selection later.

In [ ]:
metrics = [('fractionRPVs_estimatedTauR', 'refractory violations (fraction)', True),
           ('presenceRatio', 'presence ratio', False),
           ('rawAmplitude', 'raw amplitude (µV)', True),
           ('signalToNoiseRatio', 'SNR', True),
           ('percentageSpikesMissing_gaussian', '% spikes missing', False),
           ('maxDriftEstimate', 'max drift (µm)', True)]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (m, name, logx) in zip(axes.ravel(), metrics):
    for lab in ['good', 'mua', 'nonso', 'noise']:
        v = ci.loc[ci.label == lab, m].dropna()
        v = v[v > 0] if logx else v
        ax.hist(v, bins=np.logspace(np.log10(v.min()+1e-3), np.log10(v.max()+1e-3), 40) if logx else 40,
                histtype='step', lw=1.5, color=LABEL_COLORS[lab], label=lab)
    if logx: ax.set_xscale('log')
    ax.set_xlabel(name)
axes[0, 0].legend(frameon=False); sns.despine(); plt.tight_layout()

### Putative cell types from waveform duration
Trough-to-peak duration is bimodal in cortex (narrow = putative fast-spiking interneurons, broad = putative pyramidal). Whether that holds here depends on the region.

In [ ]:
g = ci[ci.label == 'good']
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(g.duration_ms, bins=np.arange(0.1, 1.35, 0.05), color=LABEL_COLORS['good'])
axes[0].set_xlabel('trough-to-peak (ms)'); axes[0].set_ylabel('n good units')
axes[1].scatter(g.duration_ms, g.fr_hz, s=12, alpha=.7, color=LABEL_COLORS['good'])
axes[1].set_yscale('log'); axes[1].set_xlabel('trough-to-peak (ms)'); axes[1].set_ylabel('firing rate (Hz)')
narrow = (g.duration_ms < 0.45).sum()
print(f'{narrow} of {len(g)} good units have trough-to-peak < 0.45 ms')
sns.despine(); plt.tight_layout()

## 2. Spike times
29 M spikes. Loading takes a few seconds; converting to seconds makes everything downstream simpler.

In [ ]:
t0 = time.time()
st, sc = io.load_spikes(DATA_DIR)
T_END = st.max()
print(f'{len(st):,} spikes in {time.time()-t0:.1f} s; recording length {T_END:.1f} s = {T_END/60:.1f} min')
print('spike times monotonic:', bool(np.all(np.diff(st) >= 0)))
print('every cluster id has metadata:', set(np.unique(sc)) <= set(ci.index))

good_ids = ci.index[ci.label == 'good'].to_numpy()
mua_ids = ci.index[ci.label == 'mua'].to_numpy()
spk = io.spikes_by_unit(st, sc, good_ids)   # dict cluster_id -> spike times (s)
print(f'{len(good_ids)} good units, {len(mua_ids)} mua units')

### Raster: good units sorted by depth, one 30 s window
Change `t_start` to look elsewhere in the session.

In [ ]:
t_start, win = 1200, 30
order = ci.loc[good_ids].sort_values('depth_um').index.to_numpy()
fig, ax = plt.subplots(figsize=(15, 8))
for row, u in enumerate(order):
    t = spk[u]; t = t[(t >= t_start) & (t < t_start + win)]
    ax.vlines(t - t_start, row, row + 0.9, lw=0.4, color='k')
ax.set_xlim(0, win); ax.set_ylim(0, len(order))
ax.set_xlabel(f'time (s) from {t_start} s'); ax.set_ylabel('good units (sorted by depth, shallow at top)')
ax.invert_yaxis(); sns.despine(); plt.tight_layout()

## 3. Population activity across the session
1 s bins for every good unit. The heatmap is z-scored per unit and smoothed over 10 s so slow state changes are visible. The last minute drops to near zero because the recording stopped at 5707.9 s.

In [ ]:
BIN = 1.0
counts, edges = io.bin_spikes(st, sc, order, BIN, t_max=T_END)
rates = counts / BIN
t_bins = edges[:-1] + BIN / 2
pop_rate = rates.mean(0)

# 10 s smoothing + z-score per unit
k = 10
sm = np.apply_along_axis(lambda r: np.convolve(r, np.ones(k)/k, mode='same'), 1, rates)
z = (sm - sm.mean(1, keepdims=True)) / (sm.std(1, keepdims=True) + 1e-9)

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True, gridspec_kw=dict(height_ratios=[4, 1]))
im = axes[0].imshow(z, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2,
                    extent=[0, T_END/60, len(order), 0])
axes[0].set_ylabel('good units (sorted by depth)'); axes[0].set_title('z-scored firing rate, 10 s smoothing')
plt.colorbar(im, ax=axes[0], pad=0.01, label='z')
axes[1].plot(t_bins/60, np.convolve(pop_rate, np.ones(k)/k, mode='same'), color='k', lw=1)
axes[1].set_ylabel('mean rate (Hz)'); axes[1].set_xlabel('session time (min)')
sns.despine(); plt.tight_layout()

### Stability: first half vs second half
Chronic recordings drift. Units whose rate changes a lot between halves are worth checking against `maxDriftEstimate` before including them in event-locked analyses.

In [ ]:
half = int(len(t_bins) // 2)
fr1, fr2 = rates[:, :half].mean(1), rates[:, half:-60].mean(1)   # drop the final minute
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(fr1, fr2, s=12, alpha=.7, color=LABEL_COLORS['good'])
lim = [1e-2, 1e2]; axes[0].plot(lim, lim, 'k--', lw=.8)
axes[0].set_xscale('log'); axes[0].set_yscale('log'); axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel('rate, first half (Hz)'); axes[0].set_ylabel('rate, second half (Hz)')
ratio = np.log2((fr2 + 1e-3) / (fr1 + 1e-3))
axes[1].scatter(ci.loc[order, 'maxDriftEstimate'], ratio, s=12, alpha=.7, color=LABEL_COLORS['good'])
axes[1].set_xscale('log'); axes[1].set_xlabel('max drift estimate (µm)'); axes[1].set_ylabel('log2(rate 2nd / 1st half)')
axes[1].axhline(0, color='k', lw=.8)
print(f'{(np.abs(ratio) > 1).sum()} of {len(order)} good units change rate by more than 2x between halves')
sns.despine(); plt.tight_layout()

## 4. Single-unit sanity checks
ISI distributions and autocorrelograms for a few example units. A clean unit has a dip at short lags (refractory period).

In [ ]:
examples = ci.loc[good_ids].sort_values('n_spikes', ascending=False).index[[0, 20, 100, 250]]
fig, axes = plt.subplots(2, len(examples), figsize=(4*len(examples), 6))
for j, u in enumerate(examples):
    t = spk[u]
    isi = np.diff(t) * 1000
    axes[0, j].hist(isi[isi < 100], bins=100, color='k')
    axes[0, j].axvline(2, color='r', lw=.8)
    axes[0, j].set_title(f'unit {u}  depth {ci.depth_um[u]:.0f} µm  {ci.fr_hz[u]:.1f} Hz', fontsize=10)
    axes[0, j].set_xlabel('ISI (ms)')
    lag, acg = io.autocorrelogram(t[:200_000], bin_ms=1, window_ms=50)
    axes[1, j].bar(lag, acg, width=1, color='k')
    axes[1, j].set_xlabel('lag (ms)')
axes[0, 0].set_ylabel('count'); axes[1, 0].set_ylabel('autocorrelogram')
sns.despine(); plt.tight_layout()

## 5. Video
Overhead IR camera. Frames are dark, so the montage below is contrast-enhanced (CLAHE). The mouse carries a headstage and tether. Objects visible inside the arena: bedding, two pellets, a round dish. The stimulus cages outside the glass are not obvious from this angle.

In [ ]:
VIDEO = DATA_DIR / 'video_compressed.mkv'
meta = io.video_metadata(VIDEO)
meta['duration_min'] = meta['duration_s'] / 60
print(meta)
print(f"video is {meta['duration_s'] - T_END:.0f} s ({(meta['duration_s'] - T_END)/60:.1f} min) longer than the ephys")

In [ ]:
import cv2
clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
mins = [2, 20, 40, 60, 80, 100]
fig, axes = plt.subplots(2, 3, figsize=(15, 9.5))
for ax, m in zip(axes.ravel(), mins):
    fr = io.read_frame(VIDEO, int(m * 60 * meta['fps']))
    ax.imshow(clahe.apply(fr), cmap='gray'); ax.set_title(f'{m} min (video time)'); ax.axis('off')
plt.tight_layout()

### Coarse video timeline
Mean brightness and frame-to-frame motion energy sampled every 10 s (cached in `cache/video_coarse_10s.npy`). Motion is fairly flat while the mouse explores, then spikes near minute 110 to 116, which is the experimenter removing the mouse after the ephys had already stopped.

In [ ]:
vc = io.coarse_video_scan(VIDEO, step_s=10, cache=CACHE / 'video_coarse_10s.npy')
vt, vmean, vmotion = vc
fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
axes[0].plot(vt/60, vmean, color='k'); axes[0].set_ylabel('mean pixel intensity')
axes[1].plot(vt/60, vmotion, color='k'); axes[1].set_ylabel('motion energy (10 s step)')
for ax in axes:
    ax.axvspan(0, T_END/60, color=LABEL_COLORS['good'], alpha=.15, label='ephys duration')
    ax.axvline(T_END/60, color=LABEL_COLORS['good'])
axes[1].set_xlabel('video time (min)'); axes[0].legend(frameon=False)
sns.despine(); plt.tight_layout()

## 6. Video-to-ephys alignment

There is no sync file, so alignment is measured from the data: video motion energy (mean absolute frame difference, full frame rate, cached in `cache/motion_energy_fullrate.npy`) is cross-correlated with the good-unit population rate. If the two clocks agree, the peak sits at zero lag in every block of the session.

In [ ]:
rows = np.load(CACHE / 'align_blocks_5min.npy')          # (block centre s, ephys - video offset s, peak r)
warp = io.load_video_warp(CACHE)
ts_v, mot = io.load_motion_energy(CACHE)                 # container video time
ts_e = io.video_to_ephys_time(ts_v, warp)
good_ids = ci.index[ci.label == 'good'].to_numpy()
stg = st[np.isin(sc, good_ids)]

def xcorr_offsets(t_video, BIN, maxlag_s):
    edges = np.arange(0, T_END, BIN)
    vm = io.bin_signal(t_video, mot, edges); pr = np.histogram(stg, edges)[0] / BIN
    ok = ~np.isnan(vm); vm, pr = vm[ok], pr[ok]
    vm = (vm - vm.mean()) / vm.std(); pr = (pr - pr.mean()) / pr.std()
    lags = np.arange(-int(maxlag_s / BIN), int(maxlag_s / BIN) + 1)
    r = np.array([np.mean(vm[L:] * pr[:len(pr) - L]) if L >= 0 else np.mean(vm[:L] * pr[-L:]) for L in lags])
    return -lags * BIN, r          # x axis = ephys time minus video time

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
ax = axes[0]
ax.scatter(rows[:, 0] / 60, rows[:, 1], c=rows[:, 2], cmap='viridis', s=40, zorder=3)
xx = np.linspace(0, T_END, 2); ax.plot(xx / 60, warp[0] + warp[1] * xx, 'k--', lw=1,
        label=f'fit: {warp[0]:+.2f} s + {warp[1]*3600:+.1f} s/hour')
ax.set_xlabel('session time (min)'); ax.set_ylabel('ephys time minus video time (s)'); ax.legend(frameon=False)
ax.set_title('offset per 5-min block (colour = peak r)')
ax = axes[1]
for tv, name in [(ts_v, 'raw video clock'), (ts_e, 'warped video clock')]:
    x, r = xcorr_offsets(tv, 1.0, 60); ax.plot(x, r, lw=1.2, label=f'{name}: peak r={r.max():.2f} at {x[np.argmax(r)]:+.0f} s')
ax.axvline(0, color='k', lw=.5); ax.set_xlabel('ephys time minus video time (s)'); ax.set_ylabel('correlation (1 s bins)')
ax.legend(frameon=False); ax.set_title('whole-session cross-correlation, motion energy vs population rate')
sns.despine(); plt.tight_layout()
print(f'effective frame rate {meta["fps"] / (1 + warp[1]):.3f} fps vs nominal {meta["fps"]:.3f}; '
      f'video frame for ephys t=1000 s is {io.ephys_to_frame(1000, warp)}')

**Result.** The offset is near zero at the start and grows linearly at about 22 s per hour, with 0.18 s residual scatter across blocks. The effective frame rate is 59.28 fps rather than the nominal 59.64. After the linear warp the whole-session peak sits at 0 s and the correlation more than doubles. Use `io.frame_to_ephys_time` for any frame-indexed tracking output.

**Confirmed against the raw recording** (`/Volumes/Research/YttriMiceInJailRecordings082626`, Open Ephys ONIX via Bonsai; details in `session_metadata.json`):
* `np2-a-clock_*.raw` holds a 250 MHz hardware tick for every ephys sample: 8333.33 ticks per sample, so the ephys clock is 30 kHz to 9 ppm (51 ms over the session). `spike_times / 30000` is accurate ephys time.
* `camera_timestamps_*.csv` logs 420,924 camera frames at a rock-steady 59.6374 fps (2 frames skipped by the camera itself). Both video files contain 418,492 frames. The 2,432 missing frames (0.578%) were dropped by the Bonsai ffmpeg/NVENC writer, which also wrote synthetic constant-rate timestamps into the container, so the drop positions are not recorded anywhere. Uniform drops predict a stretch of 1.00581 versus the fitted 1.00607, a 1.5 s disagreement at the end of the session, so drops were close to but not perfectly uniform.
* The camera clock (device nanoseconds) and the ONIX clock are independent, and `digital_inputs_*.csv` has no timestamp column, so the constant offset (0.29 s) is known only from the cross-correlation, to roughly 0.2 s.

**For future sessions:** route the camera exposure strobe into an ONIX digital input and log the ONIX clock alongside frame number, add the clock column to the digital-input CSV writer, and either lower the writer load (resolution, codec preset) or log writer drops so frame index maps exactly to camera timestamps.